# RMP Resonance Analysis in a 3D Stellarator

This tutorial demonstrates publication-quality analysis of **Resonant Magnetic Perturbation (RMP)** effects on stellarator magnetic topology.

## Physics Background

### Resonance Condition
A perturbation field component $(m, n)$ is resonant at the flux surface $\Psi_{res}$ where:
$$q(\Psi_{res}) = \frac{m}{n}$$

### Island Half-Width (Rutherford Formula)
The island half-width in normalized flux coordinate:
$$w_\Psi = 4\sqrt{\frac{|b_{mn}|}{m \cdot |dq/d\Psi|}}$$

### O-point Phase
The complex Fourier coefficient $b_{mn} = |b_{mn}| e^{i\phi_{mn}}$ sets the island O-point:
$$\theta_O = \frac{\phi_{mn}}{m} \pmod{2\pi/m}$$

### Chirikov Overlap
Two adjacent islands overlap (leading to chaos) when:
$$\sigma_{\text{Chirikov}} = \frac{w_1 + w_2}{|\Psi_2 - \Psi_1|} > 1$$

In [ ]:
import sys
sys.path.insert(0, r'D:\Repo\pyna')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

%matplotlib inline

plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 120,
    'axes.labelsize': 12,
})

## 1. Build Stellarator Equilibrium

We use `SimpleStellarartor` — an analytic model with:
- Nested circular flux surfaces centered at $(R_0, 0)$
- Linear safety factor profile: $q(\psi) = q_0 + (q_1 - q_0)\psi$
- Helical ripple perturbation to break axisymmetry

In [ ]:
from pyna.MCF.equilibrium.stellarator import simple_stellarator

eq = simple_stellarator(
    R0=3.0,          # major radius (m)
    r0=0.3,          # minor radius (m)
    B0=2.5,          # on-axis toroidal field (T)
    q0=1.5,          # safety factor on axis
    q1=4.5,          # safety factor at LCFS
    m_h=3,           # helical poloidal mode (N_fp=3 field periods)
    n_h=3,           # helical toroidal mode
    epsilon_h=0.03,  # 3% helical ripple amplitude
)

print(eq)
print(f'q range: {eq.q_of_psi(0):.2f} to {eq.q_of_psi(1):.2f}')

# Show accessible rational surfaces
print('\nAccessible rational surfaces:')
for m_t, n_t in [(2,1),(5,2),(3,1),(7,2),(4,1)]:
    psi_list = eq.resonant_psi(m_t, n_t)
    if psi_list:
        print(f'  q={m_t}/{n_t}={m_t/n_t:.3f}  psi_res={psi_list[0]:.3f}')

## 2. Poincaré Cross-Section

Since the stellarator is **not axisymmetric**, we trace field lines numerically and collect $\phi=0$ crossings to visualize flux surfaces.

In [ ]:
from pyna.MCF.visual.equilibrium import plot_nested_flux_surfaces

fig, ax = plot_nested_flux_surfaces(
    eq,
    n_fieldlines=20,
    n_turns=200,
    cmap='plasma',
    show_colorbar=True,
)
ax.set_title('Stellarator Poincaré Cross-Section', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Define RMP Field

We use a multi-mode RMP that simultaneously drives resonances at three different rational surfaces:
- $(m,n)=(2,1)$: $q=2.0$ surface at $\psi=0.167$
- $(m,n)=(5,2)$: $q=2.5$ surface at $\psi=0.333$
- $(m,n)=(3,1)$: $q=3.0$ surface at $\psi=0.500$

In [ ]:
B_rmp = 5e-3   # 5 mT = 0.2% of B0
R0_eq = eq.R0

rmp_modes = [
    (2, 1, 1.00),    # (m, n, relative amplitude)
    (5, 2, 0.60),
    (3, 1, 0.35),
]

def delta_B_rmp(R, Z, phi):
    """Multi-mode RMP: superposition of helical perturbations."""
    theta_pol = np.arctan2(Z, R - R0_eq)
    dBR, dBZ = 0.0, 0.0
    for (m_r, n_r, amp_r) in rmp_modes:
        phase = m_r * theta_pol - n_r * phi
        dBR += B_rmp * amp_r * np.cos(phase) * np.cos(theta_pol)
        dBZ += B_rmp * amp_r * np.cos(phase) * np.sin(theta_pol)
    return np.array([dBR, dBZ, 0.0])

# Verify the field at a test point
R_test = eq.R0 + 0.2
db = delta_B_rmp(R_test, 0, 0)
print(f'RMP at (R={R_test}, Z=0, phi=0): dBR={db[0]*1e3:.2f} mT, dBZ={db[1]*1e3:.2f} mT')

## 4. Compute Resonant Components and Island Widths

For each mode $(m, n)$:
1. Find resonant surface $\psi_{res}$ where $q(\psi_{res}) = m/n$
2. Sample $\delta B^\psi$ on that surface and extract $b_{mn}$ via FFT
3. Apply Rutherford formula for island half-width
4. Compute O-point phase from $\arg(b_{mn})$

In [ ]:
from pyna.MCF.visual.RMP_spectrum import ResonantComponent

dq_dpsi = eq.q1 - eq.q0  # dq/dpsi = constant for linear profile
components = []

for idx, (m_k, n_k, amp_k) in enumerate(rmp_modes):
    psi_list = eq.resonant_psi(m_k, n_k)
    if not psi_list:
        continue
    psi_res = psi_list[0]
    
    # Analytic b_mn for cosine mode: real part = B_rmp * amp_k / 2
    b_mn = B_rmp * amp_k / 2.0  # |b_mn|
    
    # Rutherford formula
    w_psi = 4.0 * np.sqrt(b_mn / (m_k * abs(dq_dpsi)))
    w_r   = w_psi * eq.r0 / (2 * np.sqrt(max(psi_res, 0.01)))
    
    # O-point at theta=pi for cos(m*theta) phase
    theta_O = np.pi / m_k  # O-point at the mode's phase
    
    comp = ResonantComponent(
        m=m_k, n=n_k,
        harmonic_order=idx+1,
        b_mn=complex(b_mn),
        psi_res=psi_res,
        q_res=float(eq.q_of_psi(psi_res)),
        half_width_psi=w_psi,
        half_width_r=w_r,
        opoint_theta=theta_O,
        xpoint_theta=theta_O + np.pi/m_k,
    )
    components.append(comp)
    
    print(f'({m_k},{n_k}): psi_res={psi_res:.3f}  q={comp.q_res:.3f}  '
          f'|b_mn|={b_mn:.3e} T  w={w_psi:.4f} psi ({w_r*100:.2f} cm)')

## 5. Chirikov Overlap Analysis

In [ ]:
print('Chirikov overlap analysis:')
for i in range(len(components) - 1):
    c1, c2 = components[i], components[i+1]
    gap   = abs(c2.psi_res - c1.psi_res)
    sigma = (c1.half_width_psi + c2.half_width_psi) / gap
    status = 'OVERLAPPING (chaos)' if sigma > 1 else 'separated'
    print(f'  ({c1.m},{c1.n}) <-> ({c2.m},{c2.n}): gap={gap:.3f}  sigma={sigma:.3f}  [{status}]')

## 6. Publication Figure

In [ ]:
from matplotlib.gridspec import GridSpec
from pyna.MCF.visual.equilibrium import ISLAND_CMAPS
from pyna.MCF.visual.RMP_spectrum import plot_island_width_bars

fig = plt.figure(figsize=(13, 6))
gs  = GridSpec(1, 2, figure=fig, width_ratios=[1, 1.3], wspace=0.4)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Left: Poincare + islands
_, ax1 = plot_nested_flux_surfaces(eq, ax=ax1, n_fieldlines=20, n_turns=150)

for ci, comp in enumerate(components):
    r_s = np.sqrt(comp.psi_res) * eq.r0
    th  = np.linspace(0, 2*np.pi, 200)
    ax1.plot(eq.R0 + r_s*np.cos(th), r_s*np.sin(th), '--',
             color=ISLAND_CMAPS[ci], lw=1.0, alpha=0.7)

plot_island_width_bars(ax1, components, eq)

patches = [mpatches.Patch(color=ISLAND_CMAPS[i], 
                          label=f'$({c.m},{c.n})$ q={c.q_res:.2f}')
           for i, c in enumerate(components)]
ax1.legend(handles=patches, loc='upper right', fontsize=8, framealpha=0.9)
ax1.set_title('Poincaré Section & Island Widths', fontsize=12)

# Right: spectrum
x = np.arange(len(components))
bars = ax2.bar(x, [abs(c.b_mn) for c in components],
               color=[ISLAND_CMAPS[i] for i in range(len(components))],
               alpha=0.82, edgecolor='white')
ax2.set_yscale('log')
ax2.set_xticks(x)
ax2.set_xticklabels([f'$({c.m},{c.n})$\nq={c.q_res:.2f}' for c in components])
ax2.set_ylabel(r'$|b_{mn}|$ (T)')
ax2.set_title('RMP Fourier Spectrum', fontsize=12)

ax_r = ax2.twinx()
ax_r.plot(x, [c.half_width_r*100 for c in components], 'D--',
          color='#E91E63', ms=9, lw=2, label=r'$w_{1/2}$ (cm)')
ax_r.set_ylabel(r'Island half-width (cm)', color='#E91E63')
ax_r.tick_params(axis='y', labelcolor='#E91E63')
ax_r.legend(loc='lower right')

fig.suptitle('Stellarator RMP Resonance Analysis', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 7. Summary

| Mode $(m,n)$ | $q_{res}$ | $\psi_{res}$ | $|b_{mn}|$ (mT) | $w_{1/2}$ (cm) |
|:---:|:---:|:---:|:---:|:---:|
| (2,1) | 2.00 | 0.167 | 2.50 | 3.00 |
| (5,2) | 2.50 | 0.333 | 1.50 | 1.04 |
| (3,1) | 3.00 | 0.500 | 0.875 | 0.84 |

**Chirikov criterion**: $\sigma < 1$ for all adjacent pairs → islands are separated, no global chaos onset at this RMP amplitude.

Increasing the RMP amplitude would eventually cause $\sigma > 1$ and stochastic field-line transport.